## preprocessing

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

In [3]:
df = pd.read_csv("../data/Feature_engineered_dataset.csv")

print("Shape:", df.shape)

df.head()

Shape: (7043, 28)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TotalCharges,Churn,TenureYears,IsNewCustomer,IsLongTermCustomer,ServiceCount,ExpectedTotalCharges,ChargeDifference,AverageMonthlyCharge,ContractTenure
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,29.85,No,0.083333,1,0,1,29.85,0.00,29.850000,Month-to-month_New
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,1889.50,No,2.833333,0,0,2,1936.30,-46.80,55.573529,One year_MidTerm
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,108.15,Yes,0.166667,1,0,2,107.70,0.45,54.075000,Month-to-month_New
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,1840.75,No,3.750000,0,0,3,1903.50,-62.75,40.905556,One year_MidTerm
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,151.65,Yes,0.166667,1,0,0,141.40,10.25,75.825000,Month-to-month_New


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Rows: 7043
Columns: 28

Missing Values:
0

Duplicate Rows:
22


In [5]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (7043, 27)
y Shape: (7043,)


In [6]:
y = y.map({
    "No": 0,
    "Yes": 1
})

print(y.value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
print("Target Distribution:")
print(y.value_counts())

print("\nTarget Percentage:")
print(y.value_counts(normalize=True) * 100)

Target Distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target Percentage:
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 27)
X_test : (1409, 27)
y_train: (5634,)
y_test : (1409,)


In [10]:
print("Training Target Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True))

Training Target Distribution:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Testing Target Distribution:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [11]:
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

print("\nNumber of Numerical Columns:", len(numerical_columns))

Numerical Columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TenureYears', 'IsNewCustomer', 'IsLongTermCustomer', 'ServiceCount', 'ExpectedTotalCharges', 'ChargeDifference', 'AverageMonthlyCharge']

Number of Numerical Columns: 11


In [12]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

print("\nNumber of Categorical Columns:", len(categorical_columns))

Categorical Columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'ContractTenure']

Number of Categorical Columns: 16


C:\Users\yashg\AppData\Local\Temp\ipykernel_28476\1385343385.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [13]:
print("Numerical Features:")
for col in numerical_columns:
    print("-", col)

print("\nCategorical Features:")
for col in categorical_columns:
    print("-", col)

Numerical Features:
- SeniorCitizen
- tenure
- MonthlyCharges
- TotalCharges
- TenureYears
- IsNewCustomer
- IsLongTermCustomer
- ServiceCount
- ExpectedTotalCharges
- ChargeDifference
- AverageMonthlyCharge

Categorical Features:
- gender
- Partner
- Dependents
- PhoneService
- MultipleLines
- InternetService
- OnlineSecurity
- OnlineBackup
- DeviceProtection
- TechSupport
- StreamingTV
- StreamingMovies
- Contract
- PaperlessBilling
- PaymentMethod
- ContractTenure


In [14]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [15]:
categorical_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_columns
        ),
        (
            "cat",
            categorical_transformer,
            categorical_columns
        )
    ]
)

In [17]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [18]:
print("Original X_train Shape:", X_train.shape)
print("Processed X_train Shape:", X_train_processed.shape)

print("\nOriginal X_test Shape:", X_test.shape)
print("Processed X_test Shape:", X_test_processed.shape)

Original X_train Shape: (5634, 27)
Processed X_train Shape: (5634, 48)

Original X_test Shape: (1409, 27)
Processed X_test Shape: (1409, 48)


In [19]:
print(type(X_train_processed))
print(type(X_test_processed))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [20]:
feature_names = preprocessor.get_feature_names_out()

print("Total Processed Features:", len(feature_names))

print("\nFirst 30 Features:")
print(feature_names[:30])

Total Processed Features: 48

First 30 Features:
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'num__TenureYears' 'num__IsNewCustomer'
 'num__IsLongTermCustomer' 'num__ServiceCount' 'num__ExpectedTotalCharges'
 'num__ChargeDifference' 'num__AverageMonthlyCharge' 'cat__gender_Male'
 'cat__Partner_Yes' 'cat__Dependents_Yes' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No phone service' 'cat__MultipleLines_Yes'
 'cat__InternetService_Fiber optic' 'cat__InternetService_No'
 'cat__OnlineSecurity_No internet service' 'cat__OnlineSecurity_Yes'
 'cat__OnlineBackup_No internet service' 'cat__OnlineBackup_Yes'
 'cat__DeviceProtection_No internet service' 'cat__DeviceProtection_Yes'
 'cat__TechSupport_No internet service' 'cat__TechSupport_Yes'
 'cat__StreamingTV_No internet service' 'cat__StreamingTV_Yes'
 'cat__StreamingMovies_No internet service']


In [21]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [22]:
X_train_processed_df.head()

,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__TenureYears,num__IsNewCustomer,num__IsLongTermCustomer,num__ServiceCount,num__ExpectedTotalCharges,num__ChargeDifference,...,cat__ContractTenure_Month-to-month_New,cat__ContractTenure_Month-to-month_ShortTerm,cat__ContractTenure_One year_LongTerm,cat__ContractTenure_One year_MidTerm,cat__ContractTenure_One year_New,cat__ContractTenure_One year_ShortTerm,cat__ContractTenure_Two year_LongTerm,cat__ContractTenure_Two year_MidTerm,cat__ContractTenure_Two year_New,cat__ContractTenure_Two year_ShortTerm
3738,-0.441773,0.102371,-0.521976,-0.262257,0.102371,-0.667628,-0.696402,0.507935,-0.253464,-0.302311,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.503635,-0.711743,-0.667628,-0.696402,-0.570530,-0.514920,0.370873,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4860,-0.441773,-0.793155,-0.809013,-0.749883,-0.793155,-0.667628,-0.696402,0.507935,-0.778067,0.936556,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3867,-0.441773,-0.263980,0.284384,-0.172722,-0.263980,-0.667628,-0.696402,1.047168,-0.170483,-0.079152,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.989374,-1.281624,1.497840,-0.696402,-1.109762,-0.989954,-0.000564,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
scaled_numerical_columns = [
    f"num__{col}" for col in numerical_columns
]

X_train_processed_df[scaled_numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
num__SeniorCitizen,5634.0,7.440898e-17,1.000089,-0.441773,-0.441773,-0.441773,-0.441773,2.263606
num__tenure,5634.0,-1.008935e-17,1.000089,-1.322329,-0.955978,-0.141863,0.916486,1.608483
num__MonthlyCharges,5634.0,-2.402527e-16,1.000089,-1.544028,-0.971198,0.184834,0.831912,1.785939
num__TotalCharges,5634.0,2.522338e-17,1.000089,-1.008922,-0.832101,-0.396845,0.674194,2.801869
num__TenureYears,5634.0,-4.035741e-17,1.000089,-1.322329,-0.955978,-0.141863,0.916486,1.608483
num__IsNewCustomer,5634.0,5.296911e-17,1.000089,-0.667628,-0.667628,-0.667628,1.497840,1.497840
num__IsLongTermCustomer,5634.0,-3.783508e-17,1.000089,-0.696402,-0.696402,-0.696402,1.435951,1.435951
num__ServiceCount,5634.0,0.000000e+00,1.000089,-1.109762,-1.109762,-0.031297,0.507935,2.125633
num__ExpectedTotalCharges,5634.0,1.135052e-17,1.000089,-1.009513,-0.833645,-0.397561,0.679491,2.744391
num__ChargeDifference,5634.0,0.000000e+00,1.000089,-5.499469,-0.425938,-0.000564,0.426478,5.533928


In [24]:
print(
    "Missing values in processed training data:",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Missing values in processed testing data:",
    X_test_processed_df.isnull().sum().sum()
)

Missing values in processed training data: 0
Missing values in processed testing data: 0


In [25]:
print("Final X_train:", X_train_processed.shape)
print("Final X_test :", X_test_processed.shape)

print("Final y_train:", y_train.shape)
print("Final y_test :", y_test.shape)

Final X_train: (5634, 48)
Final X_test : (1409, 48)
Final y_train: (5634,)
Final y_test : (1409,)


In [27]:
preprocessing_dataset = df.copy()

preprocessing_dataset.to_csv(
    "../data/Preprocessing_dataset.csv",
    index=False
)

In [28]:
import os

os.makedirs("../data/processed", exist_ok=True)

X_train_processed_df.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Processed train-test data saved successfully!")

Processed train-test data saved successfully!


In [29]:
import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully!")

Preprocessor saved successfully!
